# 07 — Noisy Student YOLO (teacher → pseudo-labels → bigger student)

**Production loop.** Every time the warehouse accumulates new
unlabeled footage, we re-run this notebook to push a fresh
Production YOLO version. The teacher is always the current
Production model; the student trains on labeled Kaggle data PLUS
pseudo-labels generated by the teacher on the new footage.

## Unlabeled data source

We use the **Kaggle Warehouse Object Detection dataset** (Howe 2023,
`machowe/warehouse-object-detection-dataset`) as the unlabeled pool.
This is a *separate* dataset from our labeled `zoya77` box-delivery
set — different warehouses, different box types, forklifts, shelving,
varied lighting conditions. Using a second, different-distribution
source is the correct Noisy Student setup: the student must generalise
across both domains, not just memorise the teacher's predictions on
the same images it was trained on.

In production this dataset is replaced by live RTSP dumps from
the warehouse cameras (`datasets/raw/pexels_warehouse/` for local
demos once footage has been captured).

## Why this matters for production

1. Our labeled dataset (Kaggle, 361 train images) caps the headroom
   of a one-shot training run. The 2-Phase ULMFiT notebook (`06`)
   already trains a strong model from those 361 images — but it
   can't get *better* without new labels.

2. Real-world unlabeled footage from a different distribution forces
   the student to learn more robust, generalisable features. This is
   the key contribution of Noisy Student (Xie et al. 2020, *Self-
   training with Noisy Student improves ImageNet classification*).

3. **Gating**: a new student is only promoted to Production if it
   beats the teacher's mAP@0.5 on the held-out Kaggle test split.
   No risk of silent regression.

## Iteration pattern

```
  v0  =  yolov8n on COCO  (untouched)
  v1  =  v0 fine-tuned on Kaggle  (notebook 00 or 06)
  v2  =  Noisy Student on v1  (this notebook)        ← current run
  v3  =  Noisy Student on v2  (re-run weekly)
   …
```

## What this notebook produces

| Artefact | Path |
|---|---|
| Teacher pseudo-labels | `data/processed/pseudo_labels/` |
| Student weights | `runs/noisy_student/weights/best.pt` |
| Teacher vs Student metrics | `runs/noisy_student/metrics.json` |
| Downloadable bundle | `bundle_noisy_student_<timestamp>.zip` |


In [ ]:
# 1. Verify we are on a T4 runtime. Noisy Student needs ~45 min on T4
#    (longer than notebook 06 because the dataset is larger after
#    adding pseudo-labels).
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__)
    if not torch.cuda.is_available():
        print('\n  WARNING: no GPU detected — Runtime → Change runtime type → T4 GPU')
    else:
        print('GPU :', torch.cuda.get_device_name(0))
        print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
except ImportError:
    print('  Torch not installed yet — cell 2 will fix that.')


In [ ]:
# 2. Clone the LOGIVISION repo, set sys.path, install training deps.
#    Same defensive setup as notebooks 00 and 06 so 'from services.*'
#    imports resolve and we don't accidentally 'pip install services'
#    from PyPI (that's an UNRELATED package).
import pathlib, os, sys
REPO_URL = 'https://github.com/Ayalem/logivision_v2'
REPO_DIR = pathlib.Path('/content/logivision_v2')
if not REPO_DIR.is_dir():
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

for p in (str(REPO_DIR), str(REPO_DIR / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)
print('sys.path[0:3] =', sys.path[:3])

assert (REPO_DIR / 'services' / 'model_server' / 'service.py').is_file(), (
    'services/model_server/service.py missing — clone is stale; re-run git clone'
)

%pip install -q ultralytics==8.3.0 kagglehub==0.3.0 pyyaml==6.0.1 opencv-python-headless==4.10.0.84
print('deps installed')


In [ ]:
# 3. Pull Kaggle credentials and download the LABELED dataset (Kaggle
#    warehouse-delivery-box). Same flow as notebook 06.
import json, os, pathlib, sys
try:
    from google.colab import userdata
    KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
    KAGGLE_KEY      = userdata.get('KAGGLE_KEY')
    assert KAGGLE_USERNAME and KAGGLE_KEY, 'Missing Colab secret'
except (ImportError, Exception) as e:
    KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME')
    KAGGLE_KEY      = os.environ.get('KAGGLE_KEY')
    assert KAGGLE_USERNAME and KAGGLE_KEY, (
        f'No Kaggle creds ({e}). Add KAGGLE_USERNAME and KAGGLE_KEY as Colab Secrets.'
    )

kj = pathlib.Path.home() / '.kaggle' / 'kaggle.json'
kj.parent.mkdir(exist_ok=True)
kj.write_text(json.dumps({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}))
os.chmod(kj, 0o600)

import kagglehub
DATASET_PATH = kagglehub.dataset_download('zoya77/warehouse-delivery-box-detection-dataset')
KAGGLE_BOX = pathlib.Path(DATASET_PATH) / 'Box Dataset'
assert KAGGLE_BOX.is_dir(), f'Unexpected layout: {list(pathlib.Path(DATASET_PATH).iterdir())}'

import prepare_kaggle_warehouse as prep
prep.ROOT = KAGGLE_BOX
prep.OUT  = pathlib.Path('data/processed/kaggle_warehouse')
prep.main()

# Scene-aware re-split — fixes the Roboflow temporal-leakage bug. See
# scripts/reshuffle_splits_by_scene.py for the audit + methodology.
import subprocess
subprocess.run([sys.executable, 'scripts/reshuffle_splits_by_scene.py'], check=True)

DATA_YAML = pathlib.Path('data/processed/kaggle_warehouse_clean/data.yaml').resolve()
print('clean kaggle data.yaml:', DATA_YAML)


In [ ]:
# 4. TEACHER = the current Production YOLO. Three options for sourcing it
#    (try in order):
#       (a) latest 'logivision-detector' Production version from MLflow
#       (b) a teacher checkpoint we uploaded to the repo (committed at
#           ml/artifacts/yolo_teacher/best.pt if present)
#       (c) yolov8n.pt from COCO (cold-start; first iteration)
import pathlib

TEACHER_CANDIDATES = [
    pathlib.Path('ml/artifacts/yolo_teacher/best.pt'),
    pathlib.Path('ml/runs/two_phase/phase2/weights/best.pt'),
    pathlib.Path('runs/two_phase/phase2/weights/best.pt'),
]
TEACHER_PT = next((p for p in TEACHER_CANDIDATES if p.is_file()), None)
if TEACHER_PT is None:
    print('No teacher checkpoint found in repo; downloading COCO-pretrained yolov8n.pt')
    TEACHER_PT = pathlib.Path('yolov8n.pt')

print('TEACHER =', TEACHER_PT)


In [ ]:
# 5. Collect UNLABELED warehouse frames from a second real-world dataset.
#
#    We use the Kaggle 'Warehouse Object Detection' dataset (Howe 2023) —
#    a completely separate source from our labeled Kaggle box-delivery set.
#    Different distribution: forklifts, pallets, shelving, varied lighting.
#
#    This is the correct Noisy Student setup: teacher pseudo-labels frames
#    from a *different* distribution; student learns generalisation across
#    both sources. In production this is replaced by live RTSP dumps from
#    the warehouse cameras.
import pathlib, shutil, json as _json, os as _os

UNLABELED_DIR = pathlib.Path('data/processed/unlabeled_warehouse/images')
UNLABELED_DIR.mkdir(parents=True, exist_ok=True)

# Download the second warehouse dataset via kagglehub
import kagglehub
DATASET2_PATH = pathlib.Path(
    kagglehub.dataset_download('machowe/warehouse-object-detection-dataset')
)
print('Dataset2 root:', DATASET2_PATH)
print('Contents:', [p.name for p in DATASET2_PATH.iterdir()])

# Copy images only — we deliberately discard the labels to simulate unlabeled data.
img_exts = {'.jpg', '.jpeg', '.png'}
n_copied = 0
for img in DATASET2_PATH.rglob('*'):
    if img.suffix.lower() in img_exts and 'images' in img.parts:
        dst = UNLABELED_DIR / f'wod_{img.stem}{img.suffix}'
        if not dst.exists():
            shutil.copy(img, dst)
            n_copied += 1

print(f'copied {n_copied} unlabeled frames from Warehouse OD dataset to {UNLABELED_DIR}')
print('These frames come from a different distribution (forklifts, shelving, varied lighting).')
print('The teacher will pseudo-label box detections; the student learns domain generalisation.')


In [ ]:
# 6. TEACHER pseudo-labels every unlabeled frame. We keep only detections
#    with conf >= PSEUDO_CONF (default 0.5) to avoid teaching the student
#    the teacher's mistakes. Labels are written in YOLO txt format.
from ultralytics import YOLO
import pathlib

PSEUDO_CONF = 0.5
LABELS_DIR = pathlib.Path('data/processed/unlabeled_warehouse/labels')
LABELS_DIR.mkdir(parents=True, exist_ok=True)

teacher = YOLO(str(TEACHER_PT))
imgs = sorted(UNLABELED_DIR.glob('*.jpg'))
print(f'pseudo-labelling {len(imgs)} frames with teacher (conf >= {PSEUDO_CONF}) ...')

n_labels = 0
n_skipped = 0
for img_path in imgs:
    results = teacher.predict(str(img_path), conf=PSEUDO_CONF, verbose=False)
    label_path = LABELS_DIR / (img_path.stem + '.txt')
    lines = []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        h, w = r.orig_shape
        for box in r.boxes:
            cls_id = int(box.cls.item())
            x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
            xc = ((x1 + x2) / 2) / w
            yc = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            lines.append(f'{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    if lines:
        label_path.write_text('\n'.join(lines))
        n_labels += 1
    else:
        n_skipped += 1
print(f'  {n_labels} frames pseudo-labelled, {n_skipped} skipped (no high-conf detections)')


In [ ]:
# 7. Build a COMBINED dataset for the student: Kaggle labeled images +
#    unlabeled-with-pseudo-labels. The student YAML uses the same class
#    list as Kaggle (3 classes: box_small / box_medium / box_large).
import shutil, yaml, pathlib

COMBINED_DIR = pathlib.Path('data/processed/noisy_student')
(COMBINED_DIR / 'images' / 'train').mkdir(parents=True, exist_ok=True)
(COMBINED_DIR / 'images' / 'val').mkdir(parents=True, exist_ok=True)
(COMBINED_DIR / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
(COMBINED_DIR / 'labels' / 'val').mkdir(parents=True, exist_ok=True)

# Copy Kaggle train + val (labeled)
KAGGLE_DIR = pathlib.Path('data/processed/kaggle_warehouse')
for split in ('train', 'val'):
    for p in (KAGGLE_DIR / 'images' / split).glob('*'):
        shutil.copy(p, COMBINED_DIR / 'images' / split / p.name)
    for p in (KAGGLE_DIR / 'labels' / split).glob('*'):
        shutil.copy(p, COMBINED_DIR / 'labels' / split / p.name)

# Add pseudo-labeled frames to the TRAIN split only.
added = 0
for img in UNLABELED_DIR.glob('*.jpg'):
    label = LABELS_DIR / (img.stem + '.txt')
    if not label.is_file():
        continue
    shutil.copy(img,   COMBINED_DIR / 'images' / 'train' / img.name)
    shutil.copy(label, COMBINED_DIR / 'labels' / 'train' / label.name)
    added += 1
print(f'added {added} pseudo-labeled frames to train split')

# Reuse Kaggle's data.yaml structure with new paths.
ns_yaml = COMBINED_DIR / 'data.yaml'
kaggle_yaml = yaml.safe_load((KAGGLE_DIR / 'data.yaml').read_text())
ns_yaml.write_text(yaml.safe_dump({
    'path':  str(COMBINED_DIR.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    kaggle_yaml['nc'],
    'names': kaggle_yaml['names'],
}))
NS_DATA_YAML = ns_yaml.resolve()
print('combined data.yaml:', NS_DATA_YAML)
print((COMBINED_DIR / 'data.yaml').read_text())


In [ ]:
# 8. Train the STUDENT — yolov8s (one size larger than the teacher
#    yolov8n) with strong noise (high mixup, mosaic, dropout via
#    higher hsv/degrees, more agressive translate/scale). This is
#    where 'Noisy Student' gets its name.
from ultralytics import YOLO
import time, pathlib

print('=' * 60)
print('NOISY STUDENT TRAINING — yolov8s on Kaggle + pseudo-labels')
print('=' * 60)
t0 = time.perf_counter()

student = YOLO('yolov8s.pt')                   # one size up from teacher
ns_results = student.train(
    data=str(NS_DATA_YAML),
    epochs=40,
    imgsz=640,
    batch=24,
    optimizer='AdamW',
    lr0=1e-3,
    cos_lr=True,
    patience=10,
    device=0,
    project='runs/noisy_student',
    name='student',
    exist_ok=True,
    verbose=False,
    plots=True,
    seed=42,
    # Noise (the Noisy Student paper section 3):
    hsv_h=0.02,
    hsv_s=0.85,
    hsv_v=0.6,
    degrees=10.0,
    translate=0.15,
    scale=0.6,
    mosaic=1.0,
    mixup=0.2,
    dropout=0.1,
)
print(f'\nstudent training wall-clock: {(time.perf_counter() - t0):.0f}s')


In [ ]:
# 9. GATING — evaluate STUDENT and TEACHER on the same held-out Kaggle
#    test split. Student is only promoted to Production if it beats the
#    teacher on mAP@0.5.
import json, pathlib

teacher_val = teacher.val(data=str(DATA_YAML), split='test', verbose=False)
student_val = student.val(data=str(DATA_YAML), split='test', verbose=False)

teacher_metrics = {
    'mAP50':    float(teacher_val.box.map50),
    'mAP50-95': float(teacher_val.box.map),
}
student_metrics = {
    'mAP50':    float(student_val.box.map50),
    'mAP50-95': float(student_val.box.map),
}

print(f'{"Metric":<12} {"Teacher":<10} {"Student":<10} {"Delta":<10}')
print('-' * 44)
for k in teacher_metrics:
    t, s = teacher_metrics[k], student_metrics[k]
    delta = s - t
    arrow = '↑' if delta > 0 else ('↓' if delta < 0 else '=')
    print(f'{k:<12} {t:<10.4f} {s:<10.4f} {delta:+.4f} {arrow}')

WIN = student_metrics['mAP50'] > teacher_metrics['mAP50']
print()
print('STUDENT WINS → ready to promote' if WIN else 'STUDENT did not beat teacher → DO NOT promote')

OUT = pathlib.Path('runs/noisy_student')
(OUT / 'metrics.json').write_text(json.dumps({
    'teacher': teacher_metrics,
    'student': student_metrics,
    'student_wins': WIN,
    'unlabeled_frames': added,
    'pseudo_conf_threshold': PSEUDO_CONF,
}, indent=2))


In [ ]:
# 10. Bundle the student weights + metrics + a small set of pseudo-label
#     audit images so a human can spot-check what the teacher labelled.
import shutil
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
bundle = pathlib.Path(f'bundle_noisy_student_{stamp}')
(bundle / 'weights').mkdir(parents=True, exist_ok=True)

# Find student best.pt (Ultralytics writes runs/noisy_student/student/weights/best.pt)
candidates = sorted(pathlib.Path('runs').glob('**/student/weights/best.pt'))
if candidates:
    shutil.copy(candidates[0], bundle / 'weights' / 'best.pt')
shutil.copy(OUT / 'metrics.json', bundle / 'metrics.json')
results_csv = next(pathlib.Path('runs').glob('**/student/results.csv'), None)
if results_csv:
    shutil.copy(results_csv, bundle / 'results_student.csv')

# 6 random pseudo-label audit images
import random
pseudo_imgs = list(UNLABELED_DIR.glob('*.jpg'))
random.seed(42)
for p in random.sample(pseudo_imgs, min(6, len(pseudo_imgs))):
    (bundle / 'pseudo_audit').mkdir(exist_ok=True)
    shutil.copy(p, bundle / 'pseudo_audit' / p.name)
    lbl = LABELS_DIR / (p.stem + '.txt')
    if lbl.is_file():
        shutil.copy(lbl, bundle / 'pseudo_audit' / lbl.name)

zip_path = shutil.make_archive(str(bundle), 'zip', bundle)
print(f'bundle: {zip_path} ({pathlib.Path(zip_path).stat().st_size // 1024} KB)')

try:
    from google.colab import files
    files.download(zip_path)
    print('download triggered')
except ImportError:
    print('(outside Colab — bundle saved to disk)')


## Next steps after Colab finishes

Only promote if `student_wins = true` in `metrics.json`.

```bash
unzip ~/Downloads/bundle_noisy_student_*.zip -d ml/runs/noisy_student/
make register-from-colab RUN=noisy_student
make worker-restart
```

Open the dashboard — the AI Model Status panel will show the new
version. The Système page will surface the teacher vs student
metrics from `metrics.json` for the soutenance defense.

## Iterating (the production loop)

Every week, after new warehouse footage has been ingested:
1. Run notebook 07 again.
2. The teacher is automatically the current Production model.
3. New footage joins the pseudo-label pool.
4. Student trains, gating evaluates, only promote if it wins.

This is the **continual improvement loop** that takes the model
past what any single labeled dataset could achieve.
